# Approach B — continuous RT-residual transfer

The cleanest continuous-RT test. **No model surgery, no training loop** — the RT
"head" is a Ridge on frozen M_pop hidden states.

**Idea:** freeze M_pop → read the hidden state at each decision → fit a population
RT model (Ridge: hidden_state → log-RT) on TRAIN subjects → the individual signal is
each trial's **standardised residual** `(logRT − predicted)/sd` = *this person is
faster / slower than the population expects*. Summarise per person → cross-task
identification (within vs across domain).

Why it should beat the 10-bin version (Approach A): continuous (no discretisation),
respects ordering, and the residual is **signed** (faster/slower) — richer for
individual differences.

Compare its within/across to Approach A: choice-only ~0.15, A (bin RT) within 0.143–0.165.

## Upload the RT-value data (one-time)

Approach B needs **choice-only text + a continuous-RT sidecar** (different from the
bin data). Upload **`output_nl_rtval.zip`** (~40 MB, all 33 tasks, complete+retest) to
`/content/drive/MyDrive/sro_minitaur/`. The setup cell unzips it.

In [ ]:
# setup
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/sro-minitaur-transfer'):
    !cd /content/sro-minitaur-transfer && git pull
else:
    !git clone https://github.com/YifeiCAO/sro-minitaur-transfer.git /content/sro-minitaur-transfer
!pip -q install peft bitsandbytes scikit-learn

RTV = '/content/drive/MyDrive/sro_minitaur/output_nl_rtval'
if not os.path.isdir(RTV + '/complete'):
    !cd /content/drive/MyDrive/sro_minitaur && unzip -oq output_nl_rtval.zip
assert os.path.isdir(RTV + '/complete'), 'upload output_nl_rtval.zip to /content/drive/MyDrive/sro_minitaur/ first'
print('RT-value tasks:', len(os.listdir(RTV + '/complete')))

In [ ]:
# 1) validate on the 11 starting-subset tasks first (~15-20 min)
!cd /content/sro-minitaur-transfer && python scripts/rt_head_transfer.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop_rt \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rtval \
    --subset starting_subset --max-seq-len 4096

In [ ]:
# 2) full run on all 33 tasks (~1-2 h; per-subject hidden-state extraction)
!cd /content/sro-minitaur-transfer && python scripts/rt_head_transfer.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop_rt \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rtval \
    --subset all --max-seq-len 4096

## How to read

Per task it prints `heldout RT R^2` — **does the Ridge predict RT from the frozen
hidden states?** R² > 0 means the hidden states carry RT-relevant information (the
foundation for a residual). Then:

```
within-domain mean top1 = ?
across-domain mean top1 = ?   (chance 0.10)
```

- **within(B) > A's 0.143–0.165** → continuous RT carries more individual signal than
  10-bin (your MSE intuition wins) → worth pushing further (a trained head / DDM).
- **within(B) ≈ A** → binning wasn't the bottleneck; RT's individual signal is just
  what it is.
- **within(B) ≈ chance but R² > 0** → the model predicts *population* RT but the
  *individual* RT residual doesn't transfer — itself a clean, interpretable result.